<a href="https://colab.research.google.com/github/Gandata/Deep_learning_project/blob/dev%2Fenc-mlp/02_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Feature Extraction

This notebook extracts Concerto Small features for S3DIS Area 5. It is designed to run on a Google Colab T4 instance.

Ensure you have access to the shared Google Drive with the preprocessed data.

### 1. Setup & Mount Drive

In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/Deep_learning_project'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin dev

%cd {REPO_DIR}

# Install uv and dependencies
!pip install uv
!uv pip install --system -e .

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/Deep_learning_project'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (133/133), done.
^C
[Errno 2] No such file or directory: '/content/Deep_learning_project'
/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 42.9 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
error: /content does not appear to be a Python project, as neither `pyproject.toml` nor `setup.py` are present in the directory


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

Mounted at /content/drive
Cloning into '/content/Deep_learning_project'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 189 (delta 101), reused 129 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 30.77 MiB | 29.81 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content/Deep_learning_project
Branch 'dev/enc-mlp' set up to track remote branch 'dev/enc-mlp' from 'origin'.
Switched to a new branch 'dev/enc-mlp'
From https://github.com/Gandata/Deep_learning_project
 * branch            dev/enc-mlp -> FETCH_HEAD
Already up to date.
dev/enc-mlp
ae860c3 (HEAD -> dev/enc-mlp, origin/dev/enc-mlp) Changes to try to parallelize preprocessing


In [2]:
# Install uv and dependencies
!pip install uv
!uv pip install --system -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 36.1 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 97 packages in 657ms
Prepared 13 packages in 12.76s
Uninstalled 2 packages in 8ms
Installed 13 packages in 307ms
 + addict==2.4.0
 + comm==0.2.3
 + configargparse==1.7.5
 + dash==4.1.0
 + deep-learning-project==0.1.0 (from file:///content/Deep_learning_project)
 + ftfy==6.3.1
 - ipywidgets==7.7.1
 + ipywidgets==8.1.8
 + jedi==0.20.0
 + open-clip-torch==3.3.0
 + open3d==0.19.0
 + pyquaternion==0.9.9
 + retrying==1.4.2
 - widgetsnbextension==3.6.10
 + widgetsnbextension==4.0.15


In [3]:
!uv sync

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 105 packages in 2ms
Prepared 99 packages in 1m 35s
Installed 99 packages in 988ms
 + addict==2.4.0
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + attrs==26.1.0
 + blinker==1.9.0
 + certifi==2026.4.22
 + charset-normalizer==3.4.7
 + click==8.3.3
 + configargparse==1.7.5
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + dash==4.1.0
 + fastjsonschema==2.21.2
 + filelock==3.29.0
 + flask==3.1.3
 + fonttools==4.62.1
 + fsspec==2026.3.0
 + ftfy==6.3.1
 + h11==0.16.0
 + hf-xet==1.4.3
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.12.2
 + idna==3.13
 + importlib-metadata==9.0.0
 + itsdangerous==2.2.0
 + jinja2==3.1.6
 + joblib==1.5.3
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + jupyter-core==5.9.1
 + kiwisolver==1.5.0
 + markdown-it-py==4.0.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + mdurl==0.1.2


### 2. Symlink Data & Checkpoints
We map the Drive folders directly into the repository so our scripts can find them.

In [4]:
# Symlink Drive data
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

### 3. Setup Hugging Face Token
Input your Hugging Face token below to allow `open_clip_torch` to download the models.

In [5]:
# If token is in colab secrets

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

In [ ]:
# If it is not on colab secrets, paste it
import getpass
import os

print("Enter your Hugging Face token (it will not be saved in the notebook):")
hf_token = getpass.getpass()

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={hf_token}\n")

### 4. Run Feature Extraction
This script will iterate over Area 5 and extract the features. It is resume-safe, so if Colab disconnects, you can just run it again.

In [6]:
# Currently runs the mockup extraction script
!uv run scripts/extract_features.py --data_dir data/s3dis_processed --out_dir features/s3dis_area5

Loading Area 5 from data/s3dis_processed...
S3DISDataset: 68 rooms loaded from areas [5]
Traceback (most recent call last):
  File "/content/Deep_learning_project/src/encoder.py", line 87, in _init_concerto_backend
    import concerto
ModuleNotFoundError: No module named 'concerto'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/Deep_learning_project/src/encoder.py", line 93, in _init_concerto_backend
    import concerto
ModuleNotFoundError: No module named 'concerto'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/Deep_learning_project/scripts/extract_features.py", line 101, in <module>
    main()
  File "/content/Deep_learning_project/scripts/extract_features.py", line 64, in main
    encoder = ConcertoEncoder(
              ^^^^^^^^^^^^^^^^
  File "/content/Deep_learning_project/src/encoder.py", line 72, in __init__
    self._init_conce

auto: notebook execution

In [59]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

PyTorch: 2.10.0+cu128
CUDA: 12.8


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!du -sh /content/drive/MyDrive/DL_Project/data/s3dis_processed

7.3G	/content/drive/MyDrive/DL_Project/data/s3dis_processed


In [7]:
!sudo update-alternatives --config python3

There are 2 choices for the alternative python3 (providing /usr/bin/python3).

  Selection    Path                 Priority   Status
------------------------------------------------------------
* 0            /usr/bin/python3.12   2         auto mode
  1            /usr/bin/python3.10   1         manual mode
  2            /usr/bin/python3.12   2         manual mode

Press <enter> to keep the current choice[*], or type selection number: 1
update-alternatives: using /usr/bin/python3.10 to provide /usr/bin/python3 (python3) in manual mode


In [8]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [10]:
!python --version

Python 3.10.12


In [3]:
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wheel]


In [5]:
!sudo apt-get update
!sudo apt-get install python3.10-dev python3.10-distutils
!python3.10 -m pip install torch==2.5.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,993 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https:/

In [11]:
!python3.10 -m pip install spconv-cu120
!python3.10 -m pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.0+cu124.html
!python3.10 -m pip install huggingface_hub timm
!python3.10 -m pip install open3d fast_pytorch_kmeans psutil addict scipy camtools natsort opencv-python trimesh gradio numpy==1.26.4

Looking in links: https://data.pyg.org/whl/torch-2.5.0+cu124.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 96.2 MB/s  0:00:00
  Using cached open3d-0.19.0-cp310-cp310-manylinux_2_31_x86_64.whl.metadata (4.3 kB)
  Using cached fast_pytorch_kmeans-0.2.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached camtools-0.1.8-py3-none-any.whl.metadata (12 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached trimesh-4.12.2-py3-none-any.whl.metadata (13 kB)
  Using cached gradio-6.14.0-py3-none-any.whl.metadata (17 kB)
  Using cached dash-4.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metada

In [14]:
!python3.10 -m pip install open3d fast_pytorch_kmeans psutil addict scipy camtools natsort opencv-python trimesh gradio numpy==1.26.4 --ignore-installed blinker

  Using cached open3d-0.19.0-cp310-cp310-manylinux_2_31_x86_64.whl.metadata (4.3 kB)
  Using cached fast_pytorch_kmeans-0.2.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached addict-2.4.0-py3-none-any.whl.metadata (1.0 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached camtools-0.1.8-py3-none-any.whl.metadata (12 kB)
  Using cached natsort-8.4.0-py3-none-any.whl.metadata (21 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached trimesh-4.12.2-py3-none-any.whl.metadata (13 kB)
  Using cached gradio-6.14.0-py3-none-any.whl.metadata (17 kB)
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached dash-4.1.0-py3-none

In [28]:
!python3.10 -m pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124
!python3.10 -m pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.0+cu124.html

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.5.0%2Bcu124-cp310-cp310-linux_x86_64.whl (908.3 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0
    Uninstalling torch-2.11.0:
      Successfully uninstalled torch-2.11.0
Looking in links: https://data.pyg.org/whl/torch-2.5.0+cu124.html


In [8]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto
%cd /content/Concerto

Cloning into '/content/Concerto'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 74 (delta 10), reused 23 (delta 9), pack-reused 42 (from 1)
Receiving objects: 100% (74/74), 2.12 MiB | 28.95 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/Concerto


In [29]:
!pip list

Package                   Version
------------------------- ---------------
addict                    2.4.0
annotated-doc             0.0.4
annotated-types           0.7.0
anyio                     4.13.0
asttokens                 3.0.1
attrs                     26.1.0
blinker                   1.9.0
brotli                    1.2.0
camtools                  0.1.8
ccimport                  0.4.4
certifi                   2026.4.22
charset-normalizer        3.4.7
click                     8.3.3
comm                      0.2.3
ConfigArgParse            1.7.5
contourpy                 1.3.2
cryptography              3.4.8
cuda-bindings             13.2.0
cuda-pathfinder           1.5.4
cuda-toolkit              13.0.2
cumm-cu120                0.4.11
cycler                    0.12.1
dash                      4.1.0
dbus-python               1.2.18
decorator                 5.2.1
distro                    1.7.0
einops                    0.8.2
exceptiongroup            1.3.1
executing        

In [33]:
!pip uninstall flash_attn

Found existing installation: flash_attn 2.8.3
Uninstalling flash_attn-2.8.3:
  Would remove:
    /usr/local/lib/python3.10/dist-packages/flash_attn-2.8.3.dist-info/*
    /usr/local/lib/python3.10/dist-packages/flash_attn/*
    /usr/local/lib/python3.10/dist-packages/flash_attn_2_cuda.cpython-310-x86_64-linux-gnu.so
    /usr/local/lib/python3.10/dist-packages/hopper/*
Proceed (Y/n)? y
  Successfully uninstalled flash_attn-2.8.3


In [26]:
!python -c "import open3d; print(open3d.__version__)"

0.19.0


In [31]:
import os

file_path = '/content/Concerto/demo/0_pca.py'
with open(file_path, 'r') as file:
    content = file.read()

# Replace the model loading line to include the custom config
old_line = 'model = concerto.load("concerto_large", repo_id="Pointcept/Concerto").cuda()'
new_line = '''custom_config = dict(enc_patch_size=[1024 for _ in range(5)], enable_flash=False)
model = concerto.load("concerto_large", repo_id="Pointcept/Concerto", custom_config=custom_config).cuda()'''

content = content.replace(old_line, new_line)

with open(file_path, 'w') as file:
    file.write(content)

In [35]:
import os

file_path = '/content/Concerto/demo/0_pca.py'
with open(file_path, 'r') as file:
    content = file.read()

# Replace the interactive Open3D window call with a headless render to image
old_draw = 'o3d.visualization.draw_geometries([pcd])'
new_draw = '''
# Headless rendering to image
vis = o3d.visualization.Visualizer()
vis.create_window(visible=False)
vis.add_geometry(pcd)
vis.poll_events()
vis.update_renderer()
vis.capture_screen_image("output_pca.png")
vis.destroy_window()
print("Saved visualization to output_pca.png")
'''

if old_draw in content:
    content = content.replace(old_draw, new_draw)
    with open(file_path, 'w') as file:
        file.write(content)
    print("Patched 0_pca.py for headless rendering.")
else:
    print("Could not find the Open3D draw command. It might have already been replaced or the script structure changed.")

Patched 0_pca.py for headless rendering.


In [39]:
%%writefile /content/Concerto/demo/0_pca.py
import concerto
import torch
import open3d as o3d
import argparse
import numpy as np

try:
    import flash_attn
except ImportError:
    flash_attn = None
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_pca_color(feat, brightness=1.25, center=True):
    u, s, v = torch.pca_lowrank(feat, center=center, q=6, niter=5)
    projection = feat @ v
    projection = projection[:, :3] * 0.6 + projection[:, 3:6] * 0.4
    min_val = projection.min(dim=-2, keepdim=True)[0]
    max_val = projection.max(dim=-2, keepdim=True)[0]
    div = torch.clamp(max_val - min_val, min=1e-6)
    color = (projection - min_val) / div * brightness
    color = color.clamp(0.0, 1.0)
    return color

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--wo_color', dest='wo_color', action='store_true', help="disable the color.")
    parser.add_argument('--wo_normal', dest='wo_normal', action='store_true', help="disable the normal.")
    args = parser.parse_args()

    concerto.utils.set_seed(6783)

    if flash_attn is not None:
        model = concerto.load("concerto_large", repo_id="Pointcept/Concerto").to(device)
    else:
        custom_config = dict(enc_patch_size=[1024 for _ in range(5)], enable_flash=False)
        model = concerto.load("concerto_large", repo_id="Pointcept/Concerto", custom_config=custom_config).to(device)

    transform = concerto.transform.default()
    point = concerto.data.load("sample1")

    if args.wo_color:
        point["color"] = np.zeros_like(point["coord"])
    if args.wo_normal:
        point["normal"] = np.zeros_like(point["coord"])

    point.pop("segment200")
    segment = point.pop("segment20")
    point["segment"] = segment
    original_coord = point["coord"].copy()
    original_color = point["color"].copy()
    point = transform(point)

    with torch.inference_mode():
        for key in point.keys():
            if isinstance(point[key], torch.Tensor) and device == "cuda":
                point[key] = point[key].cuda(non_blocking=True)

        point = model(point)

        for _ in range(2):
            assert "pooling_parent" in point.keys()
            assert "pooling_inverse" in point.keys()
            parent = point.pop("pooling_parent")
            inverse = point.pop("pooling_inverse")
            parent.feat = torch.cat([parent.feat, point.feat[inverse]], dim=-1)
            point = parent
        while "pooling_parent" in point.keys():
            assert "pooling_inverse" in point.keys()
            parent = point.pop("pooling_parent")
            inverse = point.pop("pooling_inverse")
            parent.feat = point.feat[inverse]
            point = parent

        _ = point.feat[point.inverse]
        pca_color = get_pca_color(point.feat, brightness=1.2, center=True)

    original_pca_color = pca_color[point.inverse]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(original_coord)
    pcd.colors = o3d.utility.Vector3dVector(original_pca_color.cpu().detach().numpy())

    output_path = "output_pca.ply"
    o3d.io.write_point_cloud(output_path, pcd)
    print(f"Success. 3D visualization saved to /content/Concerto/{output_path}")

Overwriting /content/Concerto/demo/0_pca.py


In [36]:
!PYTHONPATH=./ python3.10 demo/0_pca.py

/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:243: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx

In [56]:
import os

folder_path = "/content/Concerto/demo"
os.makedirs(folder_path, exist_ok=True)
file_name = os.path.join(folder_path, "see_code.py")

script_content = """
import open3d as o3d
import plotly.graph_objects as go
import numpy as np
import os

def visualize_ply_to_image(file_path, output_image="visualization_output.png"):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return

    # Load and downsample slightly for faster rendering
    pcd = o3d.io.read_point_cloud(file_path)
    if len(pcd.points) > 500000:
        pcd = pcd.voxel_down_sample(voxel_size=0.02)

    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else None

    if colors is None:
        marker_style = dict(size=1.5, opacity=0.8, color='blue')
    else:
        marker_style = dict(
            size=1.5,
            color=['rgb({},{},{})'.format(int(r*255), int(g*255), int(b*255)) for r, g, b in colors],
            opacity=0.8
        )

    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode='markers', marker=marker_style
    )])

    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=0),
        scene=dict(
            aspectmode='data',
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            zaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
        )
    )

    print(f"Saving image...")
    # Using the downgraded kaleido, we don't need extra args
    fig.write_image(output_image, width=1200, height=800)
    print(f"Successfully saved to {output_image}")

if __name__ == "__main__":
    visualize_ply_to_image("output_pca.ply", "/content/Concerto/demo/visualization_output.png")
"""

with open(file_name, "w") as f:
    f.write(script_content.strip())

print("Script updated.")

Script updated.


In [40]:
!cd /content/Concerto && PYTHONPATH=./ python3.10 demo/0_pca.py

/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/usr/local/lib/python3.10/dist-packages/spconv/pytorch/functional.py:243: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx

In [57]:
!python /content/Concerto/demo/see_code.py

Saving image...
/content/Concerto/demo/see_code.py:45: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(output_image, width=1200, height=800)
Error in sys.excepthook:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/exceptiongroup/_formatting.py", line 71, in exceptiongroup_excepthook
    sys.stderr.write("".join(traceback.format_exception(etype, value, tb)))
  File "/usr/lib/python3.10/traceback.py", line 135, in format_exception
    te = TracebackException(type(value), value, tb, limit=limit, compact=True)
  File "/usr/local/lib/python3.10/dist-packages/exceptiongroup/_formatting.py", line 96, in __init__
    self.stack = traceback.StackSummary.extract(
  File "/usr/lib/python3.10/traceback.py", line 369, in extract
    fnames.add(filename)
K

In [55]:
!pip install kaleido==0.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 16.2 MB/s  0:00:04
  Attempting uninstall: kaleido
    Found existing installation: kaleido 1.3.0
    Uninstalling kaleido-1.3.0:
      Successfully uninstalled kaleido-1.3.0


In [ ]:
!pip install open3d fast_pytorch_kmeans psutil addict scipy camtools natsort opencv-python trimesh gradio numpy==1.26.4  # currently, open3d does not support numpy 2.x

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
!pip install torch

In [15]:
!python -c "import torch; print(torch.__version__)"

2.11.0+cu130


In [ ]:
CUDA_VERSION = "12.8"

In [ ]:
TORCH_VERSION = "2.11.0"

In [ ]:
# Ensure Cuda and Pytorch are already installed in your local environment

# CUDA_VERSION: cuda version of local environment (e.g., 124), check by running 'nvcc --version'
# TORCH_VERSION: torch version of local environment (e.g., 2.5.0), check by running 'python -c "import torch; print(torch.__version__)"'
#!pip install spconv-cu${CUDA_VERSION}
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu${CUDA_VERSION}.html
!pip install git+https://github.com/Dao-AILab/flash-attention.git
!pip install huggingface_hub timm

# (optional, or directly copy the concerto folder to your project)
%cd Concerto
!python setup.py install

In [ ]:
!git clone https://github.com/Pointcept/Concerto.git /content/Concerto

In [ ]:
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 2

# Verifica que ahora diga 3.11.x
!python --version

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Write to .env
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Mapping, Union
import os
import sys
import warnings

import numpy as np
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download, login
from torch import Tensor, nn


ArrayLike = Union[np.ndarray, Tensor]
PointInput = Union[ArrayLike, Mapping[str, ArrayLike]]

load_dotenv()


def _to_tensor(value: ArrayLike, name: str) -> Tensor:
    if isinstance(value, np.ndarray):
        return torch.from_numpy(value).float()
    if torch.is_tensor(value):
        return value.float()
    raise TypeError(f"`{name}` must be a numpy array or a torch tensor.")


class ConcertoEncoder(nn.Module):
    """
    Concerto encoder wrapper for the project pipeline.

    Supported inputs:
    - tensor/ndarray of shape (N, 3), (N, 6), or (N, 9)
    - dict-like inputs with keys such as:
      - `points` or `coord` for XYZ
      - `colors` or `color` for RGB
      - `normal` or `normals` for surface normals

    Output:
    - (N, feature_dim)

    Notes:
    - this encoder always uses the official Concerto backend
    - weights can be loaded either from a local checkpoint or from Hugging Face
    """

    def __init__(
        self,
        feature_dim: int = 256,
        checkpoint_path: str | Path | None = None,
        device: str | torch.device | None = None,
        repo_id: str = "Pointcept/Concerto",
        model_name: str = "concerto_small",
        enable_flash: bool = False,
    ) -> None:
        super().__init__()

        self.feature_dim = feature_dim
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path else None
        self.repo_id = repo_id
        self.model_name = model_name
        self.enable_flash = enable_flash
        self.device = torch.device(device) if device is not None else torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        self.backbone: nn.Module | None = None
        self._concerto_transform = None

        self._init_concerto_backend()
        self.backend = "concerto"
        self.eval()
        self.to(self.device)

    def _login_hf_if_available(self) -> None:
        token = os.getenv("HF_TOKEN")
        if token and token != "your_huggingface_token_here":
            try:
                login(token=token)
            except Exception as error:
                warnings.warn(f"HF login failed: {error}")

    def _init_concerto_backend(self) -> None:
        try:
            import concerto
        except Exception:
            concerto_dir = os.getenv("CONCERTO_DIR", "/content/Concerto")
            if Path(concerto_dir).exists() and concerto_dir not in sys.path:
                sys.path.insert(0, concerto_dir)
            try:
                import concerto
            except Exception as error:
                raise ImportError(
                    "The real Concerto backend requires the official `concerto` package "
                    "and its dependencies. Install them following the official repo."
                ) from error

        self._login_hf_if_available()

        custom_config = None
        if not self.enable_flash:
            custom_config = dict(
                enc_patch_size=[1024 for _ in range(5)],
                enable_flash=False,
            )

        if self.checkpoint_path is not None:
            if not self.checkpoint_path.exists():
                raise FileNotFoundError(
                    f"Checkpoint path not found: {self.checkpoint_path}"
                )
            resolved_checkpoint = self.checkpoint_path
            print(f"Loading Concerto Small from local checkpoint: {resolved_checkpoint}")
        else:
            filename = f"{self.model_name}.pth"
            resolved_checkpoint = Path(
                hf_hub_download(
                    repo_id=self.repo_id,
                    filename=filename,
                )
            )
            print(
                "Loading Concerto Small from Hugging Face cache: "
                f"{resolved_checkpoint}"
            )

        model = concerto.model.load(
            str(resolved_checkpoint),
            custom_config=custom_config,
        )

        self.backbone = model
        self._concerto_transform = concerto.transform.default()
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

    def _extract_normal_from_mapping(
        self,
        points: Mapping[str, ArrayLike],
        coord: Tensor,
    ) -> Tensor:
        normal_value = points.get("normal")
        if normal_value is None:
            normal_value = points.get("normals")
        if normal_value is None:
            return torch.zeros_like(coord)
        normal = _to_tensor(normal_value, "normal/normals")
        if normal.shape != coord.shape:
            raise ValueError(
                f"`normal` must match coord shape {tuple(coord.shape)}, got {tuple(normal.shape)}."
            )
        return normal

    def _split_point_input(
        self,
        points: PointInput,
    ) -> tuple[Tensor, Tensor | None, Tensor | None]:
        if isinstance(points, Mapping):
            coord_value = points.get("points")
            if coord_value is None:
                coord_value = points.get("coord")
            if coord_value is None:
                raise KeyError(
                    "Point dict input must contain `points` or `coord`."
                )

            color_value = points.get("colors")
            if color_value is None:
                color_value = points.get("color")

            coord = _to_tensor(coord_value, "points/coord")
            color = None if color_value is None else _to_tensor(color_value, "colors/color")
            normal = self._extract_normal_from_mapping(points, coord)
            return coord, color, normal

        tensor = _to_tensor(points, "points")
        if tensor.shape[-1] < 3:
            raise ValueError(
                "Point inputs must have at least 3 channels for XYZ coordinates."
            )

        coord = tensor[..., :3]
        color = tensor[..., 3:6] if tensor.shape[-1] >= 6 else None
        normal = tensor[..., 6:9] if tensor.shape[-1] >= 9 else None
        return coord, color, normal

    def _prepare_concerto_point(self, points: PointInput) -> dict[str, np.ndarray]:
        if isinstance(points, Mapping):
            coord, color, normal = self._split_point_input(points)
            segment_value = points.get("segment")
            if segment_value is None:
                segment_value = points.get("label")
        else:
            coord, color, normal = self._split_point_input(points)
            segment_value = None

        if coord.ndim != 2 or coord.shape[-1] != 3:
            raise ValueError(
                "Concerto inference currently supports a single point cloud "
                "with shape (N, 3) for coordinates."
            )

        if color is None:
            color = torch.zeros_like(coord)
        if normal is None:
            normal = torch.zeros_like(coord)
        if color.max().item() <= 1.0:
            color = color * 255.0

        point = {
            "coord": coord.detach().cpu().numpy().astype(np.float32),
            "color": color.detach().cpu().numpy().astype(np.float32),
            "normal": normal.detach().cpu().numpy().astype(np.float32),
        }

        if segment_value is not None:
            segment = _to_tensor(segment_value, "segment/label").long().reshape(-1)
            point["segment"] = segment.detach().cpu().numpy()

        return point

    def _run_concerto(self, points: PointInput) -> Tensor:
        assert self.backbone is not None
        assert self._concerto_transform is not None

        point = self._prepare_concerto_point(points)
        point = self._concerto_transform(point)
        for key in point.keys():
            if isinstance(point[key], torch.Tensor):
                point[key] = point[key].to(self.device, non_blocking=True)

        with torch.no_grad():
            point = self.backbone(point)

        # Official feature recovery from the Concerto README.
        for _ in range(2):
            if "pooling_parent" not in point.keys():
                break
            parent = point.pop("pooling_parent")
            inverse = point.pop("pooling_inverse")
            parent.feat = torch.cat([parent.feat, point.feat[inverse]], dim=-1)
            point = parent

        while "pooling_parent" in point.keys():
            parent = point.pop("pooling_parent")
            inverse = point.pop("pooling_inverse")
            parent.feat = point.feat[inverse]
            point = parent

        feat = point.feat[point.inverse]
        feat = F.normalize(feat, dim=-1)

        if feat.shape[-1] != self.feature_dim:
            raise ValueError(
                f"Concerto produced feature dim {feat.shape[-1]}, but this project "
                f"currently expects {self.feature_dim}. "
                "Adjust the project config or add an explicit adapter."
            )

        return feat

    def forward(self, points: PointInput) -> Tensor:
        return self._run_concerto(points)

    @torch.no_grad()
    def encode(self, points: PointInput) -> Tensor:
        return self.forward(points)

In [ ]:
encoder = ConcertoEncoder(feature_dim=256)

xyzrgb = torch.randn(1024, 6)
xyzrgb[:, 3:6] = torch.rand(1024, 3) * 255.0
features_a = encoder(xyzrgb)

structured = {
    "points": torch.randn(512, 3),
    "colors": torch.rand(512, 3),
}
features_b = encoder(structured)

print(f"Tensor input shape:      {tuple(xyzrgb.shape)}")
print(f"Tensor output shape:     {tuple(features_a.shape)}")
print(f"Structured output shape: {tuple(features_b.shape)}")

In [ ]:
# Instalación de dependencias externas requeridas por el código de Concerto
!pip install addict yapf timm

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
!python /content/Concerto/demo/0_pca.py